# Part C — Domain Shift Measurement and Adaptation

**Group 7 | Office-Home | Art → Clipart**

This notebook compares **five model variants** on the Art→Clipart domain shift:

| # | Strategy | Target labels? | Synthetic data? |
|---|----------|:-:|:-:|
| 1 | From scratch (Part A baseline) | ✗ | ✗ |
| 2 | Feature extraction – frozen (Part A) | ✗ | ✗ |
| 3 | Fine-tuning (Part A best model) | ✗ | ✗ |
| 4 | **Target fine-tuning** | ✓ 50/class | ✗ |
| 5 | **Style-transfer augmentation** | ✗ | ✓ 30/class |

Optional bonus: DANN (domain-adversarial, no target labels).

Required figures: Grad-CAM maps (C) and t-SNE before/after adaptation (D).

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.utils import get_device, set_seed, build_domain_splits, make_loaders, SEEDS, CLASSES
from src.classifier import build_resnet_finetuned
from src.domain_adaptation import (
    target_finetuning, style_augmentation_training,
    build_summary_table, load_target_samples
)

device = get_device()
print(f'Device: {device}')

## 1. Load Part A results and test loaders

In [ ]:
# Shared test sets (must match Part A)
art_train, art_val, art_test     = build_domain_splits('Art',     seed=SEEDS[0])
clip_train, clip_val, clip_test  = build_domain_splits('Clipart', seed=SEEDS[0])
_, _, src_test_loader = make_loaders(art_train,  art_val,  art_test)
_, _, tgt_test_loader = make_loaders(clip_train, clip_val, clip_test)

# Load best Part A checkpoint (fine-tuned, seed=42)
best_part_a_state = torch.load('../checkpoints/finetuned_seed42.pt', map_location=device)
print('Loaded Part A best model')

## 2. Δ_shift: baseline measurement (no adaptation)

In [ ]:
from src.utils import evaluate

baseline_model = build_resnet_finetuned()
baseline_model.load_state_dict(best_part_a_state)
baseline_model = baseline_model.to(device)

src_acc, src_per_class = evaluate(baseline_model, src_test_loader, device)
tgt_acc, tgt_per_class = evaluate(baseline_model, tgt_test_loader, device)
delta_shift = src_acc - tgt_acc

print(f'Source accuracy (Art test):    {src_acc:.4f}')
print(f'Target accuracy (Clipart test): {tgt_acc:.4f}')
print(f'Δ_shift = {delta_shift:.4f}')

# Per-class breakdown
df_cls = pd.DataFrame({'Class': CLASSES, 'Src Acc': src_per_class, 'Tgt Acc': tgt_per_class})
df_cls['Δ'] = df_cls['Src Acc'] - df_cls['Tgt Acc']
print(df_cls.to_string(index=False))

## 3. Strategy 4 — Target domain fine-tuning

In [ ]:
tgt_ft_results = target_finetuning(
    pretrained_state=best_part_a_state,
    test_target_loader=tgt_test_loader,
    num_epochs=20,
    device=device,
    checkpoint_path='../checkpoints/target_finetuning_seed42.pt',
    seed=42,
)
print(f"Target fine-tuning → tgt_acc = {tgt_ft_results['tgt_acc']:.4f}")

## 4. Strategy 5 — Style-transfer augmentation

In [ ]:
style_aug_results = style_augmentation_training(
    source_train_samples=art_train,
    source_val_samples=art_val,
    test_source_loader=src_test_loader,
    test_target_loader=tgt_test_loader,
    num_epochs=30,
    device=device,
    checkpoint_path='../checkpoints/style_augmentation_seed42.pt',
    seed=42,
)
print(f"Style augmentation → src={style_aug_results['src_acc']:.4f}, tgt={style_aug_results['tgt_acc']:.4f}")

## 5. Summary table — all 5 model variants

In [ ]:
# Collect all results (Part A results loaded from the part_a notebook)
# If running standalone, load them from checkpoints or re-run Part A

all_results = [
    # Part A results — fill these in after running part_a notebook
    {'strategy': 'scratch',   'src_mean': 0.0, 'src_std': 0.0, 'tgt_mean': 0.0, 'tgt_std': 0.0},
    {'strategy': 'frozen',    'src_mean': 0.0, 'src_std': 0.0, 'tgt_mean': 0.0, 'tgt_std': 0.0},
    {'strategy': 'finetuned', 'src_mean': 0.0, 'src_std': 0.0, 'tgt_mean': 0.0, 'tgt_std': 0.0},
    # Part C results
    {'strategy': 'target_finetuning',    'src_acc': src_acc, 'tgt_acc': tgt_ft_results['tgt_acc']},
    {'strategy': 'style_augmentation',   'src_acc': style_aug_results['src_acc'], 'tgt_acc': style_aug_results['tgt_acc']},
]

df_summary = build_summary_table(all_results)
print(df_summary.to_string(index=False))
df_summary.to_csv('../figures/summary_table.csv', index=False)
print('Saved → figures/summary_table.csv')

## 6. Figure C — Grad-CAM attention maps

One correct + one incorrect prediction per domain.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from src.utils import get_eval_transform
import torchvision.transforms as T

model_gc = build_resnet_finetuned()
model_gc.load_state_dict(best_part_a_state)
model_gc = model_gc.to(device)
model_gc.eval()

target_layer = [model_gc.layer4[-1]]  # last block of layer4

def run_gradcam(image_path, true_label, cam_obj, device, transform):
    img_pil = Image.open(image_path).convert('RGB')
    tensor  = transform(img_pil).unsqueeze(0).to(device)
    grayscale_cam = cam_obj(input_tensor=tensor, targets=[ClassifierOutputTarget(true_label)])
    img_np  = np.array(img_pil.resize((224, 224))) / 255.0
    vis     = show_cam_on_image(img_np.astype(np.float32), grayscale_cam[0], use_rgb=True)
    pred    = model_gc(tensor).argmax(dim=1).item()
    return vis, pred

eval_transform = get_eval_transform()

with GradCAM(model=model_gc, target_layers=target_layer) as cam:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for row, (domain, domain_dir) in enumerate([('Art', 'Art'), ('Clipart', 'Clipart')]):
        found_correct = found_incorrect = False
        col = 0

        for cls_idx, cls in enumerate(CLASSES):
            cls_dir = os.path.join(DATA_ROOT, domain_dir, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if not fname.endswith('.jpg'): continue
                path = os.path.join(cls_dir, fname)
                vis, pred = run_gradcam(path, cls_idx, cam, device, eval_transform)

                if pred == cls_idx and not found_correct:
                    axes[row, col].imshow(Image.open(path).resize((224,224)))
                    axes[row, col].set_title(f'{domain}\n{cls} (correct)')
                    axes[row, col].axis('off')
                    axes[row, col+1].imshow(vis)
                    axes[row, col+1].set_title('Grad-CAM')
                    axes[row, col+1].axis('off')
                    found_correct = True
                    col += 2

                elif pred != cls_idx and not found_incorrect:
                    axes[row, col].imshow(Image.open(path).resize((224,224)))
                    axes[row, col].set_title(f'{domain}\n{cls}→{CLASSES[pred]} (wrong)')
                    axes[row, col].axis('off')
                    axes[row, col+1].imshow(vis)
                    axes[row, col+1].set_title('Grad-CAM')
                    axes[row, col+1].axis('off')
                    found_incorrect = True
                    col += 2

                if found_correct and found_incorrect: break
            if found_correct and found_incorrect: break

plt.suptitle('Grad-CAM — Correct vs. Incorrect predictions per domain', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/part_c_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/part_c_gradcam.png')

## 7. Figure D — t-SNE before and after adaptation

In [ ]:
from sklearn.manifold import TSNE
import torch.nn as nn

class FeatureExtractorWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        x = self.model.conv1(x); x = self.model.bn1(x); x = self.model.relu(x)
        x = self.model.maxpool(x)
        x = self.model.layer1(x); x = self.model.layer2(x)
        x = self.model.layer3(x); x = self.model.layer4(x)
        x = self.model.avgpool(x)
        return torch.flatten(x, 1)

def extract_features(model, loader):
    ext = FeatureExtractorWrapper(model).to(device)
    ext.eval()
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            f = ext(imgs.to(device)).cpu().numpy()
            feats.append(f); labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# Before adaptation (Part A finetuned)
before_model = build_resnet_finetuned()
before_model.load_state_dict(best_part_a_state)
before_model = before_model.to(device)

# After adaptation (target fine-tuning)
after_model = build_resnet_finetuned()
after_model.load_state_dict(torch.load('../checkpoints/target_finetuning_seed42.pt', map_location=device))
after_model = after_model.to(device)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, (label_prefix, model) in enumerate([('Before adaptation', before_model), ('After adaptation', after_model)]):
    src_f, src_l = extract_features(model, src_test_loader)
    tgt_f, tgt_l = extract_features(model, tgt_test_loader)

    all_f = np.concatenate([src_f, tgt_f])
    all_l = np.concatenate([src_l, tgt_l])
    all_d = np.array([0]*len(src_f) + [1]*len(tgt_f))

    emb = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(all_f)

    colors_cls = plt.cm.tab10(np.linspace(0, 1, len(CLASSES)))

    for i, cls in enumerate(CLASSES):
        mask = all_l == i
        axes[row, 0].scatter(emb[mask, 0], emb[mask, 1], c=[colors_cls[i]], label=cls, alpha=0.6, s=15)
    axes[row, 0].set_title(f't-SNE by class — {label_prefix}')
    if row == 0: axes[row, 0].legend(fontsize=7, markerscale=2)

    for dom, name, col in [(0, 'Art (source)', 'steelblue'), (1, 'Clipart (target)', 'coral')]:
        mask = all_d == dom
        axes[row, 1].scatter(emb[mask, 0], emb[mask, 1], c=col, label=name, alpha=0.5, s=15)
    axes[row, 1].set_title(f't-SNE by domain — {label_prefix}')
    if row == 0: axes[row, 1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/part_c_tsne_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/part_c_tsne_before_after.png')

## 8. Qualitative case study

Pick 2 examples where the unadapted model fails but the adapted model succeeds.
Explain what visual cues drove the failure and the recovery.

In [ ]:
eval_transform = get_eval_transform()

before_model.eval()
after_model.eval()

case_studies = []  # collect (path, true_label, before_pred, after_pred)

for cls_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATA_ROOT, 'Clipart', cls)
    for fname in sorted(os.listdir(cls_dir))[:20]:  # check first 20
        if not fname.endswith('.jpg'): continue
        path = os.path.join(cls_dir, fname)
        tensor = eval_transform(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
        with torch.no_grad():
            before_pred = before_model(tensor).argmax(1).item()
            after_pred  = after_model(tensor).argmax(1).item()
        if before_pred != cls_idx and after_pred == cls_idx:
            case_studies.append((path, cls_idx, before_pred, after_pred))
        if len(case_studies) >= 2: break
    if len(case_studies) >= 2: break

fig, axes = plt.subplots(len(case_studies), 1, figsize=(6, 5 * len(case_studies)))
if len(case_studies) == 1: axes = [axes]

for i, (path, true_cls, before_cls, after_cls) in enumerate(case_studies):
    axes[i].imshow(Image.open(path))
    axes[i].set_title(
        f'True: {CLASSES[true_cls]}\n'
        f'Before adaptation: {CLASSES[before_cls]} ✗\n'
        f'After adaptation:  {CLASSES[after_cls]} ✓',
        fontsize=10
    )
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('../figures/part_c_case_studies.png', dpi=150)
plt.show()
print('Saved → figures/part_c_case_studies.png')